# PlantVillage leaf-disease classification
## Comparing image embeddings, classical ML models and deep-learning fine-tuning

A complete, reproducible pipeline for the Kaggle **PlantVillage** dataset (`emmarex/plantdisease`):
about 20,600 RGB leaf images in 15 classes (pepper, potato and tomato; healthy and diseased).

**What the notebook does**

1. Cleans the data, explores it interactively (Plotly) and creates one stratified train/validation/test split.
2. Extracts image embeddings with **several pretrained backbones** (ResNet50, EfficientNet-B0, DenseNet121, ConvNeXt-Tiny, ViT-B/16).
3. Runs **all 16 baseline models on every embedding** (with imbalance handling) and selects the **best embedding** using validation scores only.
4. Tunes the best models with **Optuna** and selects the **best model**.
5. Fine-tunes a CNN (same split) and compares it with the best classical model **once** on the test set.

### Pipeline map: what every section reads and writes

| § | Step | Reads | Writes |
|---|---|---|---|
| 0 | Imports and configuration | | |
| 1 | Load image paths | image folders | |
| 2 | Cleaning | image folders | `data/meta.csv`, `data/clean.csv` |
| 3 | EDA (interactive) | `data/clean.csv` | |
| 4 | Stratified split | `data/clean.csv` | `data/train.csv`, `data/val.csv`, `data/test.csv` |
| 5 | Embeddings of every backbone | `data/{train,val,test}.csv` | `features/<emb>/X_{train,val,test}.npy`, `results/extraction_times.csv` |
| 6 | Imbalance: weights and augmented copies | `data/train.csv` | `data/train_extra.csv`, `data/augmented/*.jpg`, `features/<emb>/X_train_extra.npy` |
| 7 | 16 baselines on every embedding | `features/<emb>/*.npy` | `results/baselines_<emb>.csv`, `results/baselines_all.csv` |
| 8 | **Select the best embedding** | `results/baselines_all.csv` | `results/embedding_comparison.csv`, `config/best_embedding.json` |
| 9 | Model leaderboard on the best embedding | `results/baselines_all.csv` | |
| 10 | **Optuna: select the best model** | `features/<emb>/*.npy` | `results/tuned_models_*.csv`, `config/best_ml_params.json` |
| 11 | Final ML model (train + val) | `config/best_ml_params.json` | `models/best_ml_model.joblib` |
| 12 | Deep-learning fine-tuning | `data/*.csv`, images | `models/best_dl_<backbone>.pt`, `results/dl_history_<backbone>.csv` |
| 13 | **Test evaluation (once) and comparison** | the final ML and DL models | `results/final_results.csv`, `results/report_*.csv` |
| 14 | Conclusions and limitations | | |
| A | Optional: handcrafted features | `data/*.csv` | |

### Output folder layout

```
outputs/
  data/       meta.csv  clean.csv  train.csv  val.csv  test.csv  train_extra.csv  augmented/*.jpg
  features/   <embedding>/  X_train.npy  X_val.npy  X_test.npy  X_train_extra.npy
  results/    extraction_times.csv  baselines_<emb>.csv  baselines_all.csv  embedding_comparison.csv
              tuned_models_*.csv  dl_history_<backbone>.csv  final_results.csv  report_*.csv
  models/     best_ml_model.joblib  best_dl_<backbone>.pt
  config/     best_embedding.json  best_ml_params.json  run_info.json
```

### Requirements
`numpy pandas pillow tqdm joblib scikit-learn xgboost lightgbm catboost optuna torch torchvision plotly>=5.10 scikit-image`
(`scikit-image` is only needed for the optional appendix).

### Cost warning
Every embedding multiplies the work of section 7 (16 models x number of strategies). Start with a small run by setting, in the configuration cell,
`EMB_MODELS = ["resnet50", "efficientnet_b0"]`, `STRATEGIES = ["weighted"]` and `MODELS_TO_RUN = ["LogReg", "SVC_RBF", "LightGBM"]`, then widen the run.

## Design decisions (and why)

### 1. One 15-class classifier, not two (plant type + disease)
The folder name already encodes both pieces of information (for example `Tomato_Late_blight`). We train a single classifier on the full folder name instead of a plant-type classifier plus a disease classifier because:

* **Plant type is an easy problem.** Separating pepper, potato and tomato leaves is close to trivial, so it adds little to learn or compare.
* **Disease classes depend on the plant.** Every plant has its own "healthy" class, and a disease such as *early blight* exists in both potato and tomato but looks different, so a second-stage disease model would need the plant as input (a hierarchy, not two independent models).
* **Errors compound in a two-stage system**: a wrong plant makes the disease prediction wrong too.
* **The comparison stays clean**: one label set, one confusion matrix, one macro-F1.

The plant name is still extracted into a `plant` column and used for **analysis only**. A multi-task network with two heads is a possible extension.

### 2. Split first, then extract features
The split is made on the **list of file paths** (stratified by class) before any feature exists. The pretrained backbones learn nothing from our data, so extracting before or after would not leak labels,
but splitting first is cleaner: ML and DL must share the same files, augmentation must follow the split and touch the training part only, and everything that *does* learn from data
(`StandardScaler`, `PCA`, class weights, oversampling) is fitted on training data only, inside a `Pipeline`.

### 3. Several embeddings, chosen on validation data only
The embedding often matters more than the classifier. Every backbone is evaluated with **the same split, the same pipeline (`Scaler -> PCA -> model`), the same 16 models and the same imbalance strategies**.
The best embedding is selected with the validation macro-F1 (the mean of its top-N models, which is more stable than a single lucky model). The test set plays no part in this choice.

### 4. What is needed after the embeddings?
* **No cleaning** like the image cleaning: we only verify that the matrices have the right shape, no NaN/inf and no unexpected constant columns (section 5).
* **Scaling is done inside the pipeline.** `StandardScaler` matters for LogReg, SGD, Ridge, SVC, KNN and GaussianNB, and is harmless for trees. It is fitted on the training part of each fit only, never on the whole data (that would leak).
* **PCA (256 components)** inside the pipeline makes slow models practical and keeps the comparison fair across embeddings of different sizes.
* **L2 normalisation** is available as an optional switch (`L2_NORMALIZE`); it often helps KNN/SVC on transformer embeddings.
* **No augmentation on embeddings.** Augmentation is applied to images; the augmented copies are saved as image files and embedded by every backbone, so all backbones see the *same* extra images.

### 5. Metrics: macro-F1 is the primary metric
Class sizes range from roughly 150 to 3,200 images (ratio about 20). Accuracy hides weak performance on small classes; macro-F1 treats every class equally. We also report accuracy, weighted-F1, per-class F1 and confusion matrices.

### 6. Imbalance is handled at three levels
Stratified split + macro-F1 (sections 4 and 13), class/sample weights (strategy `weighted`) and oversampling with augmented copies (strategy `oversampled`), both computed from the **training labels only**.
Oversampling is not used inside cross-validation (copies of one image could land in both the training and validation fold), so Optuna tunes with class-balanced sample weights.

### 7. The test set is touched exactly once
Embedding choice, model choice, hyper-parameters and the best deep-learning epoch all use training and validation data only. Both final models are evaluated on the test set once, in section 13.

### 8. The deep-learning model uses the same architecture as the winning embedding
With `BACKBONE = "auto"` the network that is fine-tuned is the backbone of the best embedding, so the ML-vs-DL comparison isolates *frozen features + classical model* against *fine-tuning the same network*.

### 9. Known limitation of PlantVillage
Many images are different photos of the same leaf. A random split can place near-identical images in train and test, inflating scores (often 99%+) and shrinking gaps between models.
Exact duplicates are removed with MD5, and section 8 measures near-duplicate leakage with embedding similarity. State the limitation in the report.

## Running on Google Colab (read this first)

This version is ready for Colab. The two **setup cells** below do everything environment-specific:

1. Choose `Runtime -> Change runtime type -> T4 GPU` *before* running anything.
2. **Setup 1** mounts Google Drive (cached features and results are saved there, so they survive a disconnected session) and installs the missing packages.
3. **Setup 2** downloads PlantVillage to the fast local Colab disk and finds the folder that directly contains the 15 class folders (`ROOT_FOUND`).
4. The configuration cell uses `ROOT_FOUND` automatically, saves to `MyDrive/plantvillage_outputs` and sets `NUM_WORKERS = 2`.

After a disconnection run the two setup cells again and continue: finished steps are loaded from the Drive cache.
On the free tier the CPU is slow, so start with the small run described in the cost warning above.

In [ ]:
# ============================================================
# Colab setup 1: mount Google Drive and install the missing packages
# ============================================================
# Colab already ships torch, scikit-learn, xgboost, lightgbm and plotly; install the rest.
!pip install -q catboost optuna kagglehub scikit-image

# Drive keeps cached features/results between sessions. If the pop-up is declined,
# the notebook falls back to the temporary local disk (/content/outputs).
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# ============================================================
# Colab setup 2: download PlantVillage and locate the folder with the 15 class folders
# ============================================================
import os, shutil
from pathlib import Path
import kagglehub

try:
    # Downloaded to the local Colab disk (much faster than reading thousands of images from Drive)
    dataset_path = kagglehub.dataset_download("emmarex/plantdisease")
except Exception as e:
    print("Automatic download failed:", e)
    print("Upload your kaggle.json (Kaggle -> Settings -> API -> Create New Token) when the file picker appears.")
    from google.colab import files
    files.upload()
    kaggle_dir = os.path.expanduser("~/.kaggle")
    os.makedirs(kaggle_dir, exist_ok=True)
    shutil.copy("kaggle.json", os.path.join(kaggle_dir, "kaggle.json"))
    os.chmod(os.path.join(kaggle_dir, "kaggle.json"), 0o600)
    dataset_path = kagglehub.dataset_download("emmarex/plantdisease")
print("downloaded to:", dataset_path)

_IMG_EXT = {".jpg", ".jpeg", ".png"}

def find_root(base):
    # The dataset can contain a folder nested inside a folder of the same name.
    # We want the folder with exactly 15 sub-folders that directly hold images.
    for d in [Path(base), *Path(base).rglob("*")]:
        if d.is_dir():
            subs = [s for s in d.iterdir() if s.is_dir()]
            if len(subs) == 15 and all(any(f.suffix.lower() in _IMG_EXT for f in s.iterdir()) for s in subs):
                return d
    return None

ROOT_FOUND = find_root(dataset_path)
assert ROOT_FOUND is not None, "Could not find a folder with 15 class folders under " + str(dataset_path)
print("ROOT_FOUND =", ROOT_FOUND)
print("classes:", sorted(p.name for p in ROOT_FOUND.iterdir() if p.is_dir()))

Using Colab cache for faster access to the 'plantdisease' dataset.
downloaded to: /kaggle/input/plantdisease
ROOT_FOUND = /kaggle/input/plantdisease/PlantVillage
classes: ['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']


# PART A: Data

## 0. Imports and configuration
All switches live in this one cell. Every later cell starts with a banner that lists what it reads and what it writes.

In [ ]:
# ============================================================
# [0] Imports, configuration, reproducibility
# READS : nothing
# WRITES: nothing (creates the output folders)
# ============================================================
# Install once, in a terminal inside your virtual environment:
#   pip install numpy pandas pillow tqdm joblib scikit-learn xgboost lightgbm catboost optuna torch torchvision plotly scikit-image
import os, time, json, io, base64, hashlib, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from IPython.display import display

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

import joblib
from joblib import Parallel, delayed
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder, Normalizer, normalize
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from sklearn.linear_model import LogisticRegression, SGDClassifier, RidgeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier, BaggingClassifier,
                              AdaBoostClassifier, GradientBoostingClassifier,
                              HistGradientBoostingClassifier)
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import optuna

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
pio.templates.default = "plotly_white"

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
SEED = 42
ROOT = ROOT_FOUND                            # set by "Colab setup 2" (folder that DIRECTLY contains the 15 class folders)
# Cached files and results go to Google Drive when it is mounted, otherwise to the temporary local disk
OUT = Path("/content/drive/MyDrive/plantvillage_outputs") if Path("/content/drive/MyDrive").exists() else Path("/content/outputs")
DATA_DIR, FEAT_DIR, RES_DIR, MODEL_DIR, CFG_DIR = (OUT / d for d in ["data", "features", "results", "models", "config"])
for d in (OUT, DATA_DIR, FEAT_DIR, RES_DIR, MODEL_DIR, CFG_DIR):
    d.mkdir(parents=True, exist_ok=True)
EXT = {".jpg", ".jpeg", ".png"}
USE_CACHE = True                              # reuse cached files in OUT/ (False = recompute everything)

# Plotly renderer: leave None in VS Code / JupyterLab / Colab. Use "notebook_connected" if figures appear blank (e.g. Kaggle).
RENDERER = None
if RENDERER:
    pio.renderers.default = RENDERER

# ------------------------------------------------------------
# Embeddings (feature engineering)
# ------------------------------------------------------------
EMBEDDING_DIMS = {"resnet50": 2048, "efficientnet_b0": 1280, "densenet121": 1024,
                  "convnext_tiny": 768, "vit_b_16": 768}
EMB_MODELS = ["resnet50", "efficientnet_b0", "densenet121", "convnext_tiny", "vit_b_16"]   # which embeddings to compare
EMB_SELECTION_STRATEGY = "weighted"           # imbalance strategy used to compare embeddings
EMB_TOP_N = 3                                 # an embedding is scored by the mean macro-F1 of its N best models
IMAGE_SIZE = 224                              # all five backbones expect 224 x 224 input
BATCH_SIZE = 64                               # lower it (e.g. 32) if the GPU runs out of memory
NUM_WORKERS = 2                               # Colab runs Linux, so data-loader workers are safe
PCA_COMPONENTS = 256                          # None disables PCA (must be <= the smallest embedding size)
L2_NORMALIZE = False                          # optional L2 normalisation of every embedding before scaling
TARGET_PER_CLASS = 700                        # small classes get augmented copies up to this many training images

# ------------------------------------------------------------
# Baselines and tuning
# ------------------------------------------------------------
STRATEGIES = ["weighted", "oversampled"]      # add "none" to also measure the no-handling baseline
MODELS_TO_RUN = None                          # e.g. ["LogReg", "SVC_RBF", "LightGBM"] for a quick test; None = all 16
TUNING_STRATEGY = "weighted"                  # models are ranked on this strategy; Optuna tunes with balanced weights
TUNE_EMBEDDINGS = "best"                      # "best" = tune only on the winning embedding, "all" = tune on every embedding
TOP_K = 3                                     # how many top baseline models Optuna tunes per embedding
N_TRIALS = 20
CV_FOLDS = 3
TIMEOUT_PER_MODEL = None                      # seconds per model, None = no limit

# # ------------------------------------------------------------
# # Deep learning
# # ------------------------------------------------------------
# RUN_DL = True
# BACKBONE = "auto"                             # "auto" = backbone of the final embedding, or resnet50/resnet18/efficientnet_b0/densenet121/convnext_tiny/vit_b_16
# HEAD_EPOCHS = 3                               # phase 1: train only the new classification head
# FT_EPOCHS = 5                                 # phase 2: fine-tune the whole network
# LR_HEAD = 1e-3
# LR_FT = 1e-4

# RUN_HANDCRAFTED = False                       # optional appendix A

# ------------------------------------------------------------
# Reproducibility and device
# ------------------------------------------------------------
assert set(EMB_MODELS) <= set(EMBEDDING_DIMS), "unknown embedding in EMB_MODELS"
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = device == "cuda"                    # mixed precision only on GPU
print("device:", device)
if device == "cpu":
    print("WARNING: no GPU found. Embedding extraction and fine-tuning will be slow (try Colab/Kaggle GPU).")

IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def short(label):
    # Shorten folder names for plot axes (the dataset mixes "_", "__" and "___")
    return label.replace("___", "_").replace("__", "_")

device: cuda


## 1. Load image paths
All later steps work from a table of paths and labels. The dataset archive often contains a folder nested inside a folder of the same name;
scanning the outer one would count every image twice, so the assertion below checks that all images sit under exactly one root.

In [ ]:
# ============================================================
# [1] Build the table of image paths and labels
# READS : image folders under ROOT
# WRITES: nothing
# ============================================================
assert ROOT.exists(), f"Dataset folder not found: {ROOT.resolve()}"

paths = [p for p in ROOT.rglob("*") if p.suffix.lower() in EXT]
df = pd.DataFrame({"path": [str(p) for p in paths],
                   "label": [p.parent.name for p in paths]})
df["plant"] = df["label"].str.split("_").str[0]          # used for analysis only, never as a target

# Sanity check: every image must sit exactly one level below ROOT
parents = {p.parent.parent for p in paths}
assert len(parents) == 1, f"Images were found under several roots (nested duplicate folder?): {parents}"

print("images:", len(df), "| classes:", df["label"].nunique())
display(df.head())

images: 20638 | classes: 15


,path,label,plant
0,/kaggle/input/plantdisease/PlantVillage/Pepper...,Pepper__bell___Bacterial_spot,Pepper
1,/kaggle/input/plantdisease/PlantVillage/Pepper...,Pepper__bell___Bacterial_spot,Pepper
2,/kaggle/input/plantdisease/PlantVillage/Pepper...,Pepper__bell___Bacterial_spot,Pepper
3,/kaggle/input/plantdisease/PlantVillage/Pepper...,Pepper__bell___Bacterial_spot,Pepper
4,/kaggle/input/plantdisease/PlantVillage/Pepper...,Pepper__bell___Bacterial_spot,Pepper


## 2. Cleaning
Checks performed on every file:

1. **Integrity**: the file must open and fully decode.
2. **Size and colour mode**: decides how to resize/convert (a few files can be RGBA).
3. **MD5 hash**: finds byte-identical duplicates; duplicates across splits would leak information, so one copy of each is kept.
4. **Label conflicts**: the same image under different labels makes the label unreliable, so those images are removed.

MD5 detects only *exact* copies. Near-duplicates (same leaf, different photo) are examined in section 8.

In [ ]:
# ============================================================
# [2] Integrity, size/mode and duplicate checks -> clean table
# READS : image files
# WRITES: data/meta.csv, data/clean.csv
# ============================================================
META_FILE = DATA_DIR / "meta.csv"

def inspect_image(path):
    # Returns (width, height, mode, md5, ok) for one file
    try:
        with Image.open(path) as im:
            im.verify()                    # structural check
        with Image.open(path) as im:
            im.load()                      # forces a full decode
            width, height = im.size
            mode = im.mode
        with open(path, "rb") as f:
            md5 = hashlib.md5(f.read()).hexdigest()
        return width, height, mode, md5, True
    except Exception:
        return None, None, None, None, False

if USE_CACHE and META_FILE.exists():
    meta = pd.read_csv(META_FILE)
else:
    meta = pd.DataFrame([(p, *inspect_image(p)) for p in tqdm(df["path"], desc="inspecting")],
                        columns=["path", "w", "h", "mode", "md5", "ok"])
    meta.to_csv(META_FILE, index=False)

df = df.merge(meta, on="path")
n_total = len(df)

ok = df[df["ok"]]
n_corrupted = n_total - len(ok)

dup_mask = ok.duplicated("md5", keep=False)
conflict = ok[dup_mask].groupby("md5")["label"].nunique()
conflict_hashes = conflict[conflict > 1].index

no_conflict = ok[~ok["md5"].isin(conflict_hashes)]
n_conflict = len(ok) - len(no_conflict)

clean = no_conflict.drop_duplicates("md5").copy()
n_duplicates = len(no_conflict) - len(clean)

print(f"total images          : {n_total}")
print(f"corrupted             : {n_corrupted}")
print(f"files with a duplicate: {int(dup_mask.sum())}  (extra copies removed: {n_duplicates})")
print(f"same image, diff label: {len(conflict_hashes)} hashes ({n_conflict} files removed)")
print(f"clean images          : {len(clean)}")
print("\nImage sizes:\n", ok.groupby(["w", "h"]).size().sort_values(ascending=False).head())
print("\nColour modes:\n", ok["mode"].value_counts())

# Waterfall chart of what the cleaning removed
fig = go.Figure(go.Waterfall(
    x=["Raw images", "Corrupted", "Label conflicts", "Exact duplicates", "Clean images"],
    measure=["absolute", "relative", "relative", "relative", "total"],
    y=[n_total, -n_corrupted, -n_conflict, -n_duplicates, 0],
    text=[n_total, -n_corrupted, -n_conflict, -n_duplicates, len(clean)],
    textposition="outside"))
fig.update_layout(title="Cleaning summary (number of images)", yaxis_title="images")
fig.update_yaxes(range=[len(clean) * 0.95, n_total * 1.01])
fig.show()

clean.to_csv(DATA_DIR / "clean.csv", index=False)
print("saved", DATA_DIR / "clean.csv")

inspecting:   0%|          | 0/20638 [00:00<?, ?it/s]

total images          : 20638
corrupted             : 0
files with a duplicate: 28  (extra copies removed: 14)
same image, diff label: 0 hashes (0 files removed)
clean images          : 20624

Image sizes:
 w    h  
256  256    20638
dtype: int64

Colour modes:
 mode
RGB     20637
RGBA        1
Name: count, dtype: int64


saved /content/drive/MyDrive/plantvillage_outputs/data/clean.csv


## 3. Exploratory data analysis (interactive)
Class balance, composition, colour/brightness statistics per class and actual samples. Hover for details and click legend entries to hide or isolate groups.

In [ ]:
# ============================================================
# [3a] Class distribution and imbalance
# READS : data/clean.csv (in memory: clean)
# WRITES: nothing
# ============================================================
counts = clean["label"].value_counts().rename_axis("label").reset_index(name="count")
counts["plant"] = counts["label"].str.split("_").str[0]
imbalance_ratio = counts["count"].max() / counts["count"].min()
print(f"largest class : {counts.iloc[0]['label']} ({counts['count'].max()})")
print(f"smallest class: {counts.iloc[-1]['label']} ({counts['count'].min()})")
print(f"imbalance ratio (max/min): {imbalance_ratio:.1f}")

fig = px.bar(counts, x="count", y="label", color="plant", orientation="h", text="count",
             title=f"Images per class (imbalance ratio = {imbalance_ratio:.1f})")
fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=620, yaxis_title="")
fig.show()

# Composition: area is proportional to the number of images
fig = px.treemap(counts, path=["plant", "label"], values="count", title="Dataset composition (plant -> class)")
fig.update_traces(textinfo="label+value+percent root")
fig.show()

largest class : Tomato__Tomato_YellowLeaf__Curl_Virus (3208)
smallest class: Potato___healthy (152)
imbalance ratio (max/min): 21.1


In [ ]:
# ============================================================
# [3b] Image size check (plotted only if sizes differ)
# READS : clean (in memory)
# WRITES: nothing
# ============================================================
size_table = clean.groupby(["w", "h"]).size().reset_index(name="images")
display(size_table)
if len(size_table) > 1:
    fig = px.scatter(size_table, x="w", y="h", size="images", hover_data=["images"], title="Image sizes")
    fig.show()
else:
    print("All images share one size, so a single resize to", IMAGE_SIZE, "is enough.")

,w,h,images
0,256,256,20624


All images share one size, so a single resize to 224 is enough.


In [ ]:
# ============================================================
# [3c] Brightness, contrast and mean colour per class (sampled)
# READS : clean (in memory), image files
# WRITES: nothing
# ============================================================
PIXEL_SAMPLE_PER_CLASS = 60
sample = clean.sample(frac=1, random_state=SEED).groupby("label").head(PIXEL_SAMPLE_PER_CLASS)

rows = []
for p, lab in tqdm(zip(sample["path"], sample["label"]), total=len(sample), desc="pixel stats"):
    a = np.asarray(Image.open(p).convert("RGB"), dtype=np.float32) / 255.0   # convert() also handles RGBA
    rows.append((p, lab, a.mean(), a.std(), *a.mean(axis=(0, 1))))
pix = pd.DataFrame(rows, columns=["path", "label", "brightness", "contrast", "R", "G", "B"])
pix["file"] = pix["path"].map(lambda s: Path(s).name)

fig = px.box(pix, x="label", y="brightness", color="label", points="outliers", hover_data=["file"],
             title="Brightness per class")
fig.update_layout(showlegend=False, xaxis_tickangle=-45, height=560)
fig.show()

fig = px.scatter(pix, x="brightness", y="contrast", color="label", hover_data=["file"], opacity=0.7,
                 title="Brightness vs. contrast (each point = one image)")
fig.show()

fig = px.scatter_3d(pix, x="R", y="G", z="B", color="label", hover_data=["file"], opacity=0.6,
                    title="Mean RGB colour per image (rotate with the mouse)")
fig.update_traces(marker_size=3)
fig.update_layout(height=650)
fig.show()

pixel stats:   0%|          | 0/900 [00:00<?, ?it/s]

In [ ]:
# ============================================================
# [3d] Look at the images: one sample per class, then an explorer with a class dropdown
# READS : clean (in memory), image files
# WRITES: nothing
# ============================================================
classes_sorted = sorted(clean["label"].unique())

# One image per class
one_each = np.stack([np.asarray(Image.open(clean[clean["label"] == c].sample(1, random_state=SEED)["path"].iloc[0])
                                .convert("RGB").resize((128, 128))) for c in classes_sorted])
fig = px.imshow(one_each, facet_col=0, facet_col_wrap=5, title="One sample per class")
for i, c in enumerate(classes_sorted):
    fig.layout.annotations[i]["text"] = short(c)[:24]
fig.update_xaxes(showticklabels=False); fig.update_yaxes(showticklabels=False)
fig.update_layout(height=620)
fig.show()

# Explorer: pick a class from the dropdown to see 8 random images of it
def to_data_uri(path, size=160):
    im = Image.open(path).convert("RGB").resize((size, size))
    buf = io.BytesIO(); im.save(buf, format="JPEG", quality=80)
    return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()

N_SHOW, ROWS, COLS = 8, 2, 4
fig = make_subplots(rows=ROWS, cols=COLS, horizontal_spacing=0.01, vertical_spacing=0.03)
trace_class = []
for ci, cls in enumerate(classes_sorted):
    picks = clean[clean["label"] == cls].sample(min(N_SHOW, (clean["label"] == cls).sum()), random_state=SEED)["path"]
    for k, p in enumerate(picks):
        fig.add_trace(go.Image(source=to_data_uri(p), visible=(ci == 0)), row=k // COLS + 1, col=k % COLS + 1)
        trace_class.append(ci)

buttons = [dict(label=short(c), method="update",
                args=[{"visible": [tc == ci for tc in trace_class]}, {"title": f"Samples of: {short(c)}"}])
           for ci, c in enumerate(classes_sorted)]
fig.update_xaxes(visible=False); fig.update_yaxes(visible=False)
fig.update_layout(updatemenus=[dict(buttons=buttons, direction="down", x=0.0, y=1.18, xanchor="left")],
                  title=f"Samples of: {short(classes_sorted[0])}", height=430, margin=dict(t=90, b=10))
fig.show()

## 4. Stratified train / validation / test split
The split is performed on **file paths** before any feature is extracted, stratified by class so every split keeps the class proportions.
The three CSV files are the single source of truth for every experiment (all embeddings, ML and DL). Do not change `SEED` afterwards.

In [ ]:
# ============================================================
# [4] Stratified 70 / 15 / 15 split
# READS : data/clean.csv
# WRITES: data/train.csv, data/val.csv, data/test.csv
# ============================================================
clean = pd.read_csv(DATA_DIR / "clean.csv")

train, temp = train_test_split(clean, test_size=0.30, stratify=clean["label"], random_state=SEED)
val, test = train_test_split(temp, test_size=0.50, stratify=temp["label"], random_state=SEED)
train, val, test = (d.reset_index(drop=True) for d in (train, val, test))
print("train:", len(train), "| val:", len(val), "| test:", len(test))

# Leakage checks: no file and no identical content may appear in two splits
for a, b, name in [(train, val, "train/val"), (train, test, "train/test"), (val, test, "val/test")]:
    assert not set(a["path"]) & set(b["path"]), f"path overlap in {name}"
    assert not set(a["md5"]) & set(b["md5"]), f"duplicate content in {name}"
print("no overlap between splits")

dist = pd.DataFrame({"train": train["label"].value_counts(),
                     "val": val["label"].value_counts(),
                     "test": test["label"].value_counts()}).fillna(0).astype(int)
dist["total"] = dist.sum(axis=1)
dist["train_%"] = (dist["train"] / dist["total"] * 100).round(1)
display(dist.sort_values("total", ascending=False))

long = (dist[["train", "val", "test"]].rename_axis("label").reset_index()
        .melt(id_vars="label", var_name="split", value_name="images"))
fig = px.bar(long, x="label", y="images", color="split", barmode="group", title="Images per class in each split")
fig.update_layout(xaxis_tickangle=-45, height=520)
fig.show()

fig = px.pie(values=[len(train), len(val), len(test)], names=["train", "val", "test"], hole=0.5, title="Split sizes")
fig.show()

train.to_csv(DATA_DIR / "train.csv", index=False)
val.to_csv(DATA_DIR / "val.csv", index=False)
test.to_csv(DATA_DIR / "test.csv", index=False)
print("saved train.csv, val.csv, test.csv in", DATA_DIR)

train: 14436 | val: 3094 | test: 3094
no overlap between splits


,train,val,test,total,train_%
label,,,,,
Tomato__Tomato_YellowLeaf__Curl_Virus,2245,482,481,3208,70.0
Tomato_Bacterial_spot,1489,319,319,2127,70.0
Tomato_Late_blight,1331,285,285,1901,70.0
Tomato_Septoria_leaf_spot,1240,266,265,1771,70.0
Tomato_Spider_mites_Two_spotted_spider_mite,1173,251,252,1676,70.0
Tomato_healthy,1109,238,238,1585,70.0
Pepper__bell___healthy,1035,222,221,1478,70.0
Tomato__Target_Spot,983,210,211,1404,70.0
Potato___Early_blight,700,150,150,1000,70.0


saved train.csv, val.csv, test.csv in /content/drive/MyDrive/plantvillage_outputs/data


# PART B: Features, imbalance and embedding selection

## 5. Embeddings of every backbone
A pretrained CNN/transformer with its classification layer removed turns each image into one vector (an *embedding*). Stacking the vectors gives an ordinary table (rows = images, columns = features)
that any scikit-learn style model can use. The networks are frozen, so nothing is learned from our data in this step.

| Backbone | Embedding size |
|---|---|
| ResNet50 | 2048 |
| EfficientNet-B0 | 1280 |
| DenseNet121 | 1024 |
| ConvNeXt-Tiny | 768 |
| ViT-B/16 | 768 |

Features are extracted once per backbone and split, saved to `features/<embedding>/`, and reused by every later step. Cached files are checked against the split sizes.

In [ ]:
# ============================================================
# [5a] Backbone registry, dataset class and extraction helpers
# READS : data/train.csv, data/val.csv, data/test.csv
# WRITES: nothing
# ============================================================
train = pd.read_csv(DATA_DIR / "train.csv"); val = pd.read_csv(DATA_DIR / "val.csv"); test = pd.read_csv(DATA_DIR / "test.csv")
splits = {"train": train, "val": val, "test": test}

# Deterministic transform for every embedding and split (ImageNet mean/std, 224 x 224)
eval_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class ImageDS(Dataset):
    def __init__(self, paths, tf):
        self.paths, self.tf = list(paths), tf
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        return self.tf(Image.open(self.paths[i]).convert("RGB"))   # convert("RGB") also handles RGBA

def make_extractor(name):
    # Pretrained network whose classification head is replaced by Identity, so it returns the embedding
    if name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2); m.fc = nn.Identity()
    elif name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1); m.classifier = nn.Identity()
    elif name == "densenet121":
        m = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1); m.classifier = nn.Identity()
    elif name == "convnext_tiny":
        m = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1); m.classifier[2] = nn.Identity()
    elif name == "vit_b_16":
        m = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1); m.heads = nn.Identity()
    else:
        raise ValueError(f"unknown embedding: {name}")
    return m.eval().to(device)

@torch.no_grad()
def extract_embeddings(model, paths, tf=eval_tf):
    # shuffle=False keeps the row order identical to the CSV so labels stay aligned
    loader = DataLoader(ImageDS(paths, tf), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    feats = [model(x.to(device)).cpu().numpy() for x in tqdm(loader, leave=False)]
    return np.vstack(feats).astype(np.float32)

def feat_path(emb, part):
    # part is one of: train, val, test, train_extra
    return FEAT_DIR / emb / f"X_{part}.npy"

def feat_ok(emb, part, n_rows):
    # A cached feature file is valid only if it exists and has the expected number of rows
    f = feat_path(emb, part)
    return USE_CACHE and f.exists() and np.load(f, mmap_mode="r").shape[0] == n_rows

In [ ]:
# ============================================================
# [5b] Extract embeddings for every backbone (train / val / test)
# READS : data/{train,val,test}.csv, image files
# WRITES: features/<emb>/X_{train,val,test}.npy, results/extraction_times.csv
# ============================================================
TIMES_FILE = RES_DIR / "extraction_times.csv"
times = pd.read_csv(TIMES_FILE) if TIMES_FILE.exists() else pd.DataFrame(columns=["embedding", "split", "seconds"])

for emb in EMB_MODELS:
    (FEAT_DIR / emb).mkdir(parents=True, exist_ok=True)
    missing = [s for s in splits if not feat_ok(emb, s, len(splits[s]))]
    if not missing:
        print(f"{emb}: cached")
        continue
    net = make_extractor(emb)
    for s in missing:
        t0 = time.time()
        X = extract_embeddings(net, splits[s]["path"])
        sec = time.time() - t0
        assert X.shape == (len(splits[s]), EMBEDDING_DIMS[emb]), f"unexpected shape {X.shape} for {emb}/{s}"
        np.save(feat_path(emb, s), X)
        times = times[~((times["embedding"] == emb) & (times["split"] == s))]
        times = pd.concat([times, pd.DataFrame([dict(embedding=emb, split=s, seconds=sec)])], ignore_index=True)
        print(f"{emb}/{s}: {X.shape} in {sec:.0f}s")
    del net
    if device == "cuda":
        torch.cuda.empty_cache()
times.to_csv(TIMES_FILE, index=False)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 137MB/s]


  0%|          | 0/226 [00:00<?, ?it/s]

resnet50/train: (14436, 2048) in 58s


  0%|          | 0/49 [00:00<?, ?it/s]

resnet50/val: (3094, 2048) in 14s


... (26 outputs left out)

  0%|          | 0/49 [00:00<?, ?it/s]

vit_b_16/val: (3094, 768) in 38s


  0%|          | 0/49 [00:20<?, ?it/s]

vit_b_16/test: (3094, 768) in 60s


In [ ]:
# ============================================================
# [5c] Sanity checks on the feature matrices (no cleaning needed, only verification)
# READS : features/<emb>/X_{train,val,test}.npy
# WRITES: nothing
# ============================================================
rows = []
for emb in EMB_MODELS:
    for s in splits:
        X = np.load(feat_path(emb, s))
        rows.append(dict(embedding=emb, split=s, rows=X.shape[0], dim=X.shape[1],
                         nan_or_inf=int((~np.isfinite(X)).sum()), constant_columns=int((X.std(axis=0) == 0).sum()),
                         mean=float(X.mean()), std=float(X.std())))
checks = pd.DataFrame(rows)
display(checks.round(4))
assert checks["nan_or_inf"].sum() == 0, "NaN/inf found in an embedding"
assert (checks["rows"].values == checks["split"].map({k: len(v) for k, v in splits.items()}).values).all(), "row count mismatch"

t = times[times["embedding"].isin(EMB_MODELS)]
if len(t):
    fig = px.bar(t, x="embedding", y="seconds", color="split", barmode="stack",
                 title="Extraction time per embedding (only for features extracted in this or earlier runs)")
    fig.show()

,embedding,split,rows,dim,nan_or_inf,constant_columns,mean,std
0,resnet50,train,14436,2048,0,0,0.0811,0.2619
1,resnet50,val,3094,2048,0,0,0.0810,0.2619
2,resnet50,test,3094,2048,0,0,0.0808,0.2617
3,efficientnet_b0,train,14436,1280,0,0,0.1320,0.4319
4,efficientnet_b0,val,3094,1280,0,0,0.1307,0.4309
5,efficientnet_b0,test,3094,1280,0,0,0.1320,0.4318
6,densenet121,train,14436,1024,0,0,0.3902,0.7671
7,densenet121,val,3094,1024,0,0,0.3907,0.7678
8,densenet121,test,3094,1024,0,0,0.3893,0.7665
9,convnext_tiny,train,14436,768,0,0,0.0002,0.5871


## 6. Class imbalance: weights and augmented copies
Both techniques use the **training labels only**; validation and test sets are never resampled or reweighted.

* **Class/sample weights** (`weighted`): each training sample is weighted inversely to its class frequency.
* **Oversampling with augmentation** (`oversampled`): classes with fewer than `TARGET_PER_CLASS` training images get extra augmented copies (random crop, flips, rotation, colour jitter).
  The copies are generated **once**, saved as image files and embedded by **every** backbone, so all embeddings are compared on identical data.

In [ ]:
# ============================================================
# [6a] Label encoding and class weights (TRAIN only)
# READS : train / val / test (in memory)
# WRITES: nothing
# ============================================================
le = LabelEncoder().fit(train["label"])
CLASS_NAMES = list(le.classes_)
N_CLASSES = len(CLASS_NAMES)
SHORT_NAMES = [short(c) for c in CLASS_NAMES]
y_train, y_val, y_test = (le.transform(d["label"]) for d in (train, val, test))

cw = compute_class_weight("balanced", classes=np.arange(N_CLASSES), y=y_train)     # one weight per class
sw_train = compute_sample_weight("balanced", y_train)                                # one weight per training sample

w_df = pd.DataFrame({"class": SHORT_NAMES, "weight": cw, "train_images": np.bincount(y_train)})
fig = px.bar(w_df.sort_values("weight"), x="weight", y="class", orientation="h", hover_data=["train_images"],
             title="Balanced class weights (rare classes get larger weights)")
fig.update_layout(height=560, yaxis_title="")
fig.show()

In [ ]:
# ============================================================
# [6b] Create the augmented copies of the small classes (once, saved as images)
# READS : data/train.csv (in memory), image files
# WRITES: data/augmented/*.jpg, data/train_extra.csv
# ============================================================
AUG_DIR = DATA_DIR / "augmented"
EXTRA_CSV = DATA_DIR / "train_extra.csv"

# Random augmentation applied to PIL images (the result is saved, then embedded like any other image)
pil_aug = transforms.Compose([
    transforms.RandomResizedCrop((256, 256), scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
])

if USE_CACHE and EXTRA_CSV.exists():
    extra = pd.read_csv(EXTRA_CSV)
else:
    AUG_DIR.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(SEED)
    torch.manual_seed(SEED)
    rows = []
    for lab, n in train["label"].value_counts().items():
        if n < TARGET_PER_CLASS:
            src = train.loc[train["label"] == lab, "path"].values
            picks = rng.choice(src, size=TARGET_PER_CLASS - n, replace=True)        # sample with replacement
            for k, p in enumerate(picks):
                out_path = AUG_DIR / f"{short(lab)}_{k:04d}.jpg"
                pil_aug(Image.open(p).convert("RGB")).save(out_path, quality=95)
                rows.append((str(out_path), lab, p))
    extra = pd.DataFrame(rows, columns=["path", "label", "source_path"])
    extra.to_csv(EXTRA_CSV, index=False)

y_extra = le.transform(extra["label"]) if len(extra) else np.empty(0, dtype=int)
y_train_os = np.concatenate([y_train, y_extra])                                       # labels of the oversampled training set
print("augmented copies:", len(extra), "| oversampled train size:", len(y_train_os))

counts_os = pd.DataFrame({
    "original": pd.Series(y_train).map(dict(enumerate(SHORT_NAMES))).value_counts(),
    "after oversampling": pd.Series(y_train_os).map(dict(enumerate(SHORT_NAMES))).value_counts()})
long = counts_os.rename_axis("class").reset_index().melt(id_vars="class", var_name="version", value_name="images")
fig = px.bar(long, x="class", y="images", color="version", barmode="group", title="Training images per class")
fig.update_layout(xaxis_tickangle=-45, height=520)
fig.show()

augmented copies: 1069 | oversampled train size: 15505


In [ ]:
# ============================================================
# [6c] Embed the augmented copies with every backbone
# READS : data/train_extra.csv, data/augmented/*.jpg
# WRITES: features/<emb>/X_train_extra.npy
# ============================================================
for emb in EMB_MODELS:
    if feat_ok(emb, "train_extra", len(extra)):
        print(f"{emb}: cached")
        continue
    if len(extra):
        net = make_extractor(emb)
        X_extra = extract_embeddings(net, extra["path"])          # deterministic transform: the augmentation is already in the files
        del net
        if device == "cuda":
            torch.cuda.empty_cache()
    else:
        X_extra = np.empty((0, EMBEDDING_DIMS[emb]), dtype=np.float32)
    np.save(feat_path(emb, "train_extra"), X_extra)
    print(f"{emb}: {X_extra.shape}")

  0%|          | 0/17 [00:00<?, ?it/s]

resnet50: (1069, 2048)


  0%|          | 0/17 [00:00<?, ?it/s]

efficientnet_b0: (1069, 1280)


  0%|          | 0/17 [00:00<?, ?it/s]

densenet121: (1069, 1024)


  0%|          | 0/17 [00:00<?, ?it/s]

convnext_tiny: (1069, 768)


  0%|          | 0/17 [00:00<?, ?it/s]

vit_b_16: (1069, 768)


## 7. Sixteen baseline models on every embedding
Every model lives in the same pipeline: `[L2 normalisation] -> StandardScaler -> PCA -> model`. For **each embedding** and each strategy in `STRATEGIES` all models are trained on the training data
and scored on the **validation** set (macro-F1 is the ranking metric). The test set is not used here. Results are cached per embedding in `results/baselines_<emb>.csv`
(delete a file to recompute that embedding).

* `weighted`: balanced sample weights passed to `fit` (KNN has no weight support, so it runs unweighted).
* `oversampled`: the training table plus the augmented copies of the small classes.
* `none`: no imbalance handling (add it to `STRATEGIES` for an ablation).

In [ ]:
# ============================================================
# [7a] Model zoo, pipeline builder, feature loader and helpers
# READS : nothing
# WRITES: nothing
# ============================================================
MODEL_ZOO = {
    "LogReg":               (LogisticRegression,            dict(max_iter=2000)),
    "SGD":                  (SGDClassifier,                 dict(random_state=SEED)),
    "Ridge":                (RidgeClassifier,               dict()),
    "KNN":                  (KNeighborsClassifier,          dict()),
    "SVC_RBF":              (SVC,                           dict(kernel="rbf")),
    "DecisionTree":         (DecisionTreeClassifier,        dict(random_state=SEED)),
    "RandomForest":         (RandomForestClassifier,        dict(n_jobs=-1, random_state=SEED)),
    "ExtraTrees":           (ExtraTreesClassifier,          dict(n_jobs=-1, random_state=SEED)),
    "BaggingTree":          (BaggingClassifier,             dict(n_jobs=-1, random_state=SEED)),
    "AdaBoost":             (AdaBoostClassifier,            dict(random_state=SEED)),

    "HistGradientBoosting": (HistGradientBoostingClassifier, dict(random_state=SEED)),
    "GaussianNB":           (GaussianNB,                    dict()),
    "XGBoost":              (XGBClassifier,                 dict(n_jobs=-1, random_state=SEED, tree_method="hist")),
    "LightGBM":             (LGBMClassifier,                dict(n_jobs=-1, random_state=SEED, verbose=-1)),
    "CatBoost":             (CatBoostClassifier,            dict(iterations=300, verbose=0, random_seed=SEED, thread_count=-1)),
}

#  "GradientBoosting":     (GradientBoostingClassifier,    dict(random_state=SEED)),
NO_SAMPLE_WEIGHT = {"KNN"}      # models whose fit() does not accept sample_weight

def build_pipeline(name, **params):
    # Every transformer is fitted inside fit(), so it only ever sees the training data of that call
    cls, defaults = MODEL_ZOO[name]
    steps = []
    if L2_NORMALIZE:
        steps.append(("l2", Normalizer()))
    steps.append(("scaler", StandardScaler()))
    if PCA_COMPONENTS:
        steps.append(("pca", PCA(n_components=PCA_COMPONENTS, svd_solver="randomized", random_state=SEED)))
    steps.append(("model", cls(**{**defaults, **params})))
    return Pipeline(steps)

def fit_pipeline(pipe, X, y, sample_weight=None):
    if sample_weight is None:
        pipe.fit(X, y)
    else:
        pipe.fit(X, y, model__sample_weight=sample_weight)
    return pipe

def predict_labels(pipe, X):
    # CatBoost returns a 2-D array for multi-class problems, so flatten everything to 1-D ints
    return np.asarray(pipe.predict(X)).ravel().astype(int)

def load_features(emb):
    # Loads every feature matrix of one embedding from disk
    F = {s: np.load(feat_path(emb, s)) for s in ("train", "val", "test")}
    F["extra"] = np.load(feat_path(emb, "train_extra"))
    F["train_os"] = np.vstack([F["train"], F["extra"]])
    return F

def strategy_data(strategy, name, F):
    # Returns (X, y, sample_weight) of the training data for an imbalance strategy
    if strategy == "none":
        return F["train"], y_train, None
    if strategy == "weighted":
        return F["train"], y_train, (None if name in NO_SAMPLE_WEIGHT else sw_train)
    if strategy == "oversampled":
        return F["train_os"], y_train_os, None
    raise ValueError(f"unknown strategy: {strategy}")

In [ ]:
# ============================================================
# [7b] Run all baselines on every embedding (the slowest ML step; cached per embedding)
# READS : features/<emb>/*.npy
# WRITES: results/baselines_<emb>.csv, results/baselines_all.csv
# ============================================================
def run_baselines(emb):
    cache = RES_DIR / f"baselines_{emb}.csv"
    if USE_CACHE and cache.exists():
        print(f"{emb}: loaded cached results ({cache.name})")
        return pd.read_csv(cache)
    F = load_features(emb)
    records = []
    for strategy in STRATEGIES:
        for name in tqdm(MODELS_TO_RUN or list(MODEL_ZOO), desc=f"{emb} [{strategy}]", leave=False):
            Xf, yf, sw = strategy_data(strategy, name, F)
            try:
                pipe = build_pipeline(name)
                t0 = time.time(); fit_pipeline(pipe, Xf, yf, sw); fit_sec = time.time() - t0
                t0 = time.time(); pred = predict_labels(pipe, F["val"]); pred_sec = time.time() - t0
                records.append(dict(model=name, strategy=strategy,
                                    accuracy=accuracy_score(y_val, pred),
                                    macro_f1=f1_score(y_val, pred, average="macro"),
                                    weighted_f1=f1_score(y_val, pred, average="weighted"),
                                    fit_sec=fit_sec, pred_sec=pred_sec, error=""))
            except Exception as e:                      # keep going if one model fails
                records.append(dict(model=name, strategy=strategy, accuracy=np.nan, macro_f1=np.nan,
                                    weighted_f1=np.nan, fit_sec=np.nan, pred_sec=np.nan, error=str(e)[:200]))
    out = pd.DataFrame(records)
    out.insert(0, "embedding", emb)
    out.to_csv(cache, index=False)
    return out

baseline_all = pd.concat([run_baselines(emb) for emb in EMB_MODELS], ignore_index=True)
baseline_all.to_csv(RES_DIR / "baselines_all.csv", index=False)
print("baseline runs:", len(baseline_all), "| failed:", int(baseline_all["error"].fillna("").ne("").sum()))

resnet50 [weighted]:   0%|          | 0/15 [00:00<?, ?it/s]

resnet50 [oversampled]:   0%|          | 0/15 [00:00<?, ?it/s]

efficientnet_b0 [weighted]:   0%|          | 0/15 [00:00<?, ?it/s]

efficientnet_b0 [oversampled]:   0%|          | 0/15 [00:00<?, ?it/s]

densenet121 [weighted]:   0%|          | 0/15 [00:00<?, ?it/s]

densenet121 [oversampled]:   0%|          | 0/15 [00:00<?, ?it/s]

convnext_tiny [weighted]:   0%|          | 0/15 [00:00<?, ?it/s]

convnext_tiny [oversampled]:   0%|          | 0/15 [00:00<?, ?it/s]

vit_b_16 [weighted]:   0%|          | 0/15 [00:00<?, ?it/s]

vit_b_16 [oversampled]:   0%|          | 0/15 [00:00<?, ?it/s]

baseline runs: 150 | failed: 0


In [ ]:
# ============================================================
# [7c] Overall leaderboard (embedding x model x strategy)
# READS : baseline_all (in memory)
# WRITES: nothing
# ============================================================
board = baseline_all.dropna(subset=["macro_f1"]).sort_values("macro_f1", ascending=False).reset_index(drop=True)
display(board.head(25).round(4))

failed = baseline_all[baseline_all["error"].fillna("").ne("")]
if len(failed):
    print("Runs that failed:")
    display(failed[["embedding", "model", "strategy", "error"]])

,embedding,model,strategy,accuracy,macro_f1,weighted_f1,fit_sec,pred_sec,error
0,densenet121,SVC_RBF,weighted,0.9809,0.9811,0.9809,13.7690,7.8322,
1,convnext_tiny,SVC_RBF,oversampled,0.9806,0.9781,0.9806,12.5494,6.3578,
2,densenet121,SVC_RBF,oversampled,0.9809,0.9779,0.9809,12.5956,6.8262,
3,convnext_tiny,SVC_RBF,weighted,0.9796,0.9772,0.9796,11.7957,5.9494,
4,vit_b_16,SVC_RBF,oversampled,0.9774,0.9768,0.9773,14.0351,8.5355,
5,vit_b_16,SVC_RBF,weighted,0.9774,0.9763,0.9774,11.8421,7.9904,
6,vit_b_16,LogReg,weighted,0.9706,0.9698,0.9705,5.7223,0.0601,
7,densenet121,LogReg,weighted,0.9703,0.9695,0.9704,6.6780,0.0496,
8,convnext_tiny,LogReg,weighted,0.9719,0.9687,0.9720,8.8893,0.0326,
9,convnext_tiny,LogReg,oversampled,0.9725,0.9678,0.9726,8.4875,0.0433,


## 8. Select the best embedding
Each embedding is scored on the **validation** set, using the models trained with `EMB_SELECTION_STRATEGY`:
the score is the **mean macro-F1 of its `EMB_TOP_N` best models** (more stable than a single lucky model). We also show the best single model, the mean over all models, the embedding size and the extraction time.
The winner is saved to `config/best_embedding.json`; every later section reads the features of this embedding.

In [ ]:
# ============================================================
# [8a] Compare embeddings and pick the best one
# READS : baseline_all (in memory), results/extraction_times.csv
# WRITES: results/embedding_comparison.csv, config/best_embedding.json
# ============================================================
sel = baseline_all[(baseline_all["strategy"] == EMB_SELECTION_STRATEGY)].dropna(subset=["macro_f1"])
assert len(sel), f"no baseline results for strategy '{EMB_SELECTION_STRATEGY}'; add it to STRATEGIES"
times = pd.read_csv(TIMES_FILE) if TIMES_FILE.exists() else pd.DataFrame(columns=["embedding", "split", "seconds"])

rank_rows = []
for emb, g in sel.groupby("embedding"):
    g = g.sort_values("macro_f1", ascending=False)
    rank_rows.append(dict(embedding=emb, dim=EMBEDDING_DIMS[emb],
                          best_model=g.iloc[0]["model"], best_macro_f1=g.iloc[0]["macro_f1"],
                          topN_mean_macro_f1=g.head(EMB_TOP_N)["macro_f1"].mean(),
                          all_models_mean_macro_f1=g["macro_f1"].mean(),
                          extraction_sec=times.loc[times["embedding"] == emb, "seconds"].sum()))
emb_rank = pd.DataFrame(rank_rows).sort_values("topN_mean_macro_f1", ascending=False).reset_index(drop=True)
display(emb_rank.round(4))
emb_rank.to_csv(RES_DIR / "embedding_comparison.csv", index=False)

BEST_EMB = emb_rank.loc[0, "embedding"]
with open(CFG_DIR / "best_embedding.json", "w") as f:
    json.dump({"embedding": BEST_EMB, "selection_strategy": EMB_SELECTION_STRATEGY, "top_n": EMB_TOP_N,
               "score_topN_mean_macro_f1": float(emb_rank.loc[0, "topN_mean_macro_f1"])}, f, indent=2)
print("BEST EMBEDDING:", BEST_EMB)

# Ranking bar chart
fig = px.bar(emb_rank, x="embedding", y=["topN_mean_macro_f1", "best_macro_f1", "all_models_mean_macro_f1"], barmode="group",
             title=f"Validation macro-F1 per embedding ({EMB_SELECTION_STRATEGY} strategy)")
fig.update_yaxes(range=[max(0, emb_rank["all_models_mean_macro_f1"].min() - 0.05), 1.0])
fig.show()

# Heatmap: which model works best with which embedding
pivot = sel.pivot_table(index="model", columns="embedding", values="macro_f1")
fig = px.imshow(pivot, text_auto=".3f", aspect="auto", color_continuous_scale="Blues",
                title="Validation macro-F1: model x embedding")
fig.update_layout(height=620)
fig.show()

# Quality vs. size and cost
fig = px.scatter(emb_rank, x="dim", y="topN_mean_macro_f1", size="extraction_sec", color="embedding", hover_data=["best_model"],
                 title="Embedding size vs. quality (bubble size = extraction time)")
fig.show()

,embedding,dim,best_model,best_macro_f1,topN_mean_macro_f1,all_models_mean_macro_f1,extraction_sec
0,densenet121,1024,SVC_RBF,0.9811,0.9685,0.8619,85.3979
1,convnext_tiny,768,SVC_RBF,0.9772,0.9673,0.8508,102.1920
2,vit_b_16,768,SVC_RBF,0.9763,0.9652,0.8424,275.1315
3,efficientnet_b0,1280,SVC_RBF,0.9636,0.9487,0.8253,91.2939
4,resnet50,2048,SVC_RBF,0.9118,0.9058,0.7655,106.8323


BEST EMBEDDING: densenet121


In [ ]:
# ============================================================
# [8b] Inspect the winning embedding: explained variance and a 2D/3D map
# READS : features/<BEST_EMB>/X_train.npy
# WRITES: nothing
# ============================================================
F_best = load_features(BEST_EMB)
X_tr_b = F_best["train"]

scaler_viz = StandardScaler().fit(X_tr_b)
n_viz = min(512, X_tr_b.shape[1])
pca_viz = PCA(n_components=n_viz, svd_solver="randomized", random_state=SEED).fit(scaler_viz.transform(X_tr_b))
cum = np.cumsum(pca_viz.explained_variance_ratio_)

k = min(PCA_COMPONENTS or 256, len(cum))
fig = px.line(x=np.arange(1, len(cum) + 1), y=cum,
              labels={"x": "number of PCA components", "y": "cumulative explained variance"},
              title=f"How much information PCA keeps ({BEST_EMB})")
fig.add_vline(x=k, line_dash="dash", annotation_text=f"{k} components = {cum[k - 1]:.1%}")
fig.show()

# Map of the embedding (random sample of the training set)
viz_idx = np.random.default_rng(SEED).choice(len(X_tr_b), size=min(4000, len(X_tr_b)), replace=False)
Z = pca_viz.transform(scaler_viz.transform(X_tr_b[viz_idx]))[:, :3]
emb_map = pd.DataFrame(Z, columns=["PC1", "PC2", "PC3"])
emb_map["label"] = train["label"].values[viz_idx]
emb_map["file"] = train["path"].map(lambda s: Path(s).name).values[viz_idx]

px.scatter(emb_map, x="PC1", y="PC2", color="label", hover_data=["file"], opacity=0.7,
           title=f"{BEST_EMB} map (PCA, 2D): well separated groups mean easy classes").show()
fig = px.scatter_3d(emb_map, x="PC1", y="PC2", z="PC3", color="label", hover_data=["file"], opacity=0.6,
                    title=f"{BEST_EMB} map (PCA, 3D)")
fig.update_traces(marker_size=3); fig.update_layout(height=650)
fig.show()

In [ ]:
# ============================================================
# [8c] Near-duplicate leakage check (PlantVillage has many photos of the same leaf)
# READS : features/<BEST_EMB>/X_{train,val,test}.npy
# WRITES: nothing
# ============================================================
# For every validation/test image: cosine similarity to the most similar TRAIN image.
def max_cosine_to_train(Xq, Xref, chunk=1000):
    Xq_n, Xref_n = normalize(Xq), normalize(Xref)
    best = np.empty(len(Xq_n), dtype=np.float32)
    for i in range(0, len(Xq_n), chunk):
        best[i:i + chunk] = (Xq_n[i:i + chunk] @ Xref_n.T).max(axis=1)
    return best

sim_val = max_cosine_to_train(F_best["val"], F_best["train"])
sim_test = max_cosine_to_train(F_best["test"], F_best["train"])
leak = pd.DataFrame({"split": ["val"] * len(sim_val) + ["test"] * len(sim_test),
                     "max_cosine_to_train": np.concatenate([sim_val, sim_test])})

for thr in (0.90, 0.95):
    print(f"share of test images with a train image above cosine {thr}: {(sim_test > thr).mean():.1%}")

fig = px.histogram(leak, x="max_cosine_to_train", color="split", barmode="overlay", nbins=60, opacity=0.6,
                   marginal="box", title=f"Similarity of each val/test image to its nearest train image ({BEST_EMB})")
fig.show()
print("A large mass near 1.0 means near-identical images exist across splits; mention this in the report.")

share of test images with a train image above cosine 0.9: 84.6%
share of test images with a train image above cosine 0.95: 11.6%


A large mass near 1.0 means near-identical images exist across splits; mention this in the report.


# PART C: Models

## 9. Model leaderboard on the best embedding
Same results as section 7, filtered to the selected embedding: models, imbalance strategies and training time side by side.

In [ ]:
# ============================================================
# [9] Baselines of the best embedding (interactive)
# READS : baseline_all (in memory), BEST_EMB
# WRITES: nothing
# ============================================================
bd = baseline_all[(baseline_all["embedding"] == BEST_EMB)].dropna(subset=["macro_f1"])
order = bd.groupby("model")["macro_f1"].max().sort_values(ascending=False).index.tolist()
display(bd.sort_values("macro_f1", ascending=False).reset_index(drop=True).round(4))

fig = px.bar(bd, x="model", y="macro_f1", color="strategy", barmode="group", hover_data=["accuracy", "fit_sec"],
             category_orders={"model": order}, title=f"Validation macro-F1 per model and strategy ({BEST_EMB})")
fig.update_yaxes(range=[max(0, bd["macro_f1"].min() - 0.03), 1.0])
fig.update_layout(xaxis_tickangle=-40, height=520)
fig.show()

fig = px.scatter(bd, x="fit_sec", y="macro_f1", color="model", symbol="strategy", size="accuracy",
                 hover_data=["accuracy", "pred_sec"], log_x=True, title="Quality vs. training time (log scale)")
fig.update_layout(height=520)
fig.show()

pivot = bd.pivot_table(index="model", columns="strategy", values="macro_f1").loc[order]
fig = px.imshow(pivot, text_auto=".3f", aspect="auto", color_continuous_scale="Blues",
                title="Macro-F1 heatmap: model x strategy")
fig.show()

,embedding,model,strategy,accuracy,macro_f1,weighted_f1,fit_sec,pred_sec,error
0,densenet121,SVC_RBF,weighted,0.9809,0.9811,0.9809,13.7690,7.8322,
1,densenet121,SVC_RBF,oversampled,0.9809,0.9779,0.9809,12.5956,6.8262,
2,densenet121,LogReg,weighted,0.9703,0.9695,0.9704,6.6780,0.0496,
3,densenet121,LogReg,oversampled,0.9709,0.9668,0.9710,5.8395,0.0600,
4,densenet121,SGD,weighted,0.9586,0.9549,0.9589,9.5446,0.0327,
5,densenet121,KNN,oversampled,0.9538,0.9481,0.9534,2.5473,0.8375,
6,densenet121,KNN,weighted,0.9535,0.9474,0.9531,2.4090,0.7580,
7,densenet121,SGD,oversampled,0.9580,0.9468,0.9581,9.0609,0.0376,
8,densenet121,CatBoost,weighted,0.9518,0.9450,0.9520,336.8135,0.0729,
9,densenet121,CatBoost,oversampled,0.9493,0.9408,0.9490,345.5849,0.0747,


## 10. Hyper-parameter tuning with Optuna: select the best model
For each embedding in scope (`TUNE_EMBEDDINGS = "best"` tunes only the winning embedding, `"all"` tunes every embedding), the top `TOP_K` models
(ranked by validation macro-F1 on `TUNING_STRATEGY`) are tuned. The objective is the mean macro-F1 of a stratified K-fold cross-validation on the **training set only**;
inside each fold the sample weights are recomputed and the whole pipeline is refitted, so nothing leaks between folds.
The tuned models are compared on the validation set and the best **(embedding, model, parameters)** combination is selected.

In [ ]:
# ============================================================
# [10a] Search spaces and the cross-validated objective
# READS : nothing
# WRITES: nothing
# ============================================================
def space_for(name, t):
    # Hyper-parameter search space of each model (t is an Optuna trial)
    if name == "LogReg":
        return dict(C=t.suggest_float("C", 1e-3, 1e2, log=True))
    if name == "SGD":
        return dict(alpha=t.suggest_float("alpha", 1e-6, 1e-2, log=True),
                    loss=t.suggest_categorical("loss", ["hinge", "log_loss", "modified_huber"]))
    if name == "Ridge":
        return dict(alpha=t.suggest_float("alpha", 1e-2, 1e3, log=True))
    if name == "KNN":
        return dict(n_neighbors=t.suggest_int("n_neighbors", 1, 25),
                    weights=t.suggest_categorical("weights", ["uniform", "distance"]),
                    p=t.suggest_categorical("p", [1, 2]))
    if name == "SVC_RBF":
        return dict(C=t.suggest_float("C", 1e-1, 1e3, log=True),
                    gamma=t.suggest_float("gamma", 1e-5, 1e-1, log=True))
    if name == "DecisionTree":
        return dict(max_depth=t.suggest_int("max_depth", 3, 40),
                    min_samples_leaf=t.suggest_int("min_samples_leaf", 1, 20),
                    criterion=t.suggest_categorical("criterion", ["gini", "entropy"]))
    if name in ("RandomForest", "ExtraTrees"):
        return dict(n_estimators=t.suggest_int("n_estimators", 100, 500),
                    max_depth=t.suggest_int("max_depth", 5, 60),
                    min_samples_leaf=t.suggest_int("min_samples_leaf", 1, 5),
                    max_features=t.suggest_categorical("max_features", ["sqrt", "log2"]))
    if name == "BaggingTree":
        return dict(n_estimators=t.suggest_int("n_estimators", 20, 100),
                    max_samples=t.suggest_float("max_samples", 0.5, 1.0),
                    max_features=t.suggest_float("max_features", 0.3, 1.0))
    if name == "AdaBoost":
        return dict(n_estimators=t.suggest_int("n_estimators", 50, 400),
                    learning_rate=t.suggest_float("learning_rate", 0.01, 2.0, log=True))
    if name == "GradientBoosting":
        return dict(n_estimators=t.suggest_int("n_estimators", 50, 200),
                    learning_rate=t.suggest_float("learning_rate", 0.03, 0.3, log=True),
                    max_depth=t.suggest_int("max_depth", 2, 5),
                    subsample=t.suggest_float("subsample", 0.6, 1.0))
    if name == "HistGradientBoosting":
        return dict(learning_rate=t.suggest_float("learning_rate", 0.03, 0.3, log=True),
                    max_iter=t.suggest_int("max_iter", 100, 400),
                    max_leaf_nodes=t.suggest_int("max_leaf_nodes", 15, 63),
                    l2_regularization=t.suggest_float("l2_regularization", 1e-3, 10.0, log=True))
    if name == "GaussianNB":
        return dict(var_smoothing=t.suggest_float("var_smoothing", 1e-12, 1e-3, log=True))
    if name == "XGBoost":
        return dict(n_estimators=t.suggest_int("n_estimators", 100, 500),
                    learning_rate=t.suggest_float("learning_rate", 0.03, 0.3, log=True),
                    max_depth=t.suggest_int("max_depth", 3, 8),
                    subsample=t.suggest_float("subsample", 0.6, 1.0),
                    colsample_bytree=t.suggest_float("colsample_bytree", 0.4, 1.0),
                    reg_lambda=t.suggest_float("reg_lambda", 1e-2, 10.0, log=True))
    if name == "LightGBM":
        return dict(n_estimators=t.suggest_int("n_estimators", 100, 500),
                    learning_rate=t.suggest_float("learning_rate", 0.03, 0.3, log=True),
                    num_leaves=t.suggest_int("num_leaves", 15, 63),
                    subsample=t.suggest_float("subsample", 0.6, 1.0), subsample_freq=1,
                    colsample_bytree=t.suggest_float("colsample_bytree", 0.4, 1.0),
                    reg_lambda=t.suggest_float("reg_lambda", 1e-2, 10.0, log=True))
    if name == "CatBoost":
        return dict(iterations=t.suggest_int("iterations", 200, 600),
                    learning_rate=t.suggest_float("learning_rate", 0.03, 0.3, log=True),
                    depth=t.suggest_int("depth", 4, 8),
                    l2_leaf_reg=t.suggest_float("l2_leaf_reg", 1.0, 10.0, log=True))
    raise KeyError(name)

def cv_macro_f1(name, params, X, y, n_splits=CV_FOLDS):
    # Mean macro-F1 over stratified folds of the TRAINING data; sample weights are recomputed per fold
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    scores = []
    for tr_idx, va_idx in skf.split(X, y):
        pipe = build_pipeline(name, **params)
        sw = None if name in NO_SAMPLE_WEIGHT else compute_sample_weight("balanced", y[tr_idx])
        fit_pipeline(pipe, X[tr_idx], y[tr_idx], sw)
        scores.append(f1_score(y[va_idx], predict_labels(pipe, X[va_idx]), average="macro"))
    return float(np.mean(scores))

In [ ]:
# ============================================================
# [10b] Tune the top-K models of every embedding in scope
# READS : features/<emb>/*.npy, baseline_all (in memory)
# WRITES: results/tuned_models_<scope>.csv
# ============================================================
TUNED_FILE = RES_DIR / f"tuned_models_{TUNE_EMBEDDINGS}.csv"
tune_targets = [BEST_EMB] if TUNE_EMBEDDINGS == "best" else list(EMB_MODELS)
studies = {}

def tune_embedding(emb):
    F = load_features(emb)
    ranking = (baseline_all[(baseline_all["embedding"] == emb) & (baseline_all["strategy"] == TUNING_STRATEGY)]
               .dropna(subset=["macro_f1"]).sort_values("macro_f1", ascending=False))
    rows = []
    for name in ranking["model"].tolist()[:TOP_K]:
        def objective(trial, name=name):
            return cv_macro_f1(name, space_for(name, trial), F["train"], y_train)
        study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED),
                                    study_name=f"{emb}/{name}")
        study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT_PER_MODEL, show_progress_bar=True)
        studies[(emb, name)] = study

        # Fit on the full training set with the best parameters and score on validation
        pipe = build_pipeline(name, **study.best_params)
        fit_pipeline(pipe, F["train"], y_train, None if name in NO_SAMPLE_WEIGHT else sw_train)
        pred = predict_labels(pipe, F["val"])
        rows.append(dict(embedding=emb, model=name, cv_macro_f1=study.best_value,
                         val_macro_f1=f1_score(y_val, pred, average="macro"),
                         val_accuracy=accuracy_score(y_val, pred), best_params=study.best_params))
    return rows

if USE_CACHE and TUNED_FILE.exists():
    tuned_df = pd.read_csv(TUNED_FILE)
    tuned_df["best_params"] = tuned_df["best_params"].apply(json.loads)
    print("loaded cached tuning results (delete", TUNED_FILE.name, "to retune); Optuna plots are unavailable for cached runs")
else:
    all_rows = []
    for emb in tune_targets:
        print("tuning on embedding:", emb)
        all_rows += tune_embedding(emb)
    tuned_df = pd.DataFrame(all_rows)
    tuned_df.assign(best_params=tuned_df["best_params"].apply(json.dumps)).to_csv(TUNED_FILE, index=False)

tuned_df = tuned_df.sort_values("val_macro_f1", ascending=False).reset_index(drop=True)
display(tuned_df)

loaded cached tuning results (delete tuned_models_best.csv to retune); Optuna plots are unavailable for cached runs


,embedding,model,cv_macro_f1,val_macro_f1,val_accuracy,best_params
0,densenet121,SVC_RBF,0.981567,0.985891,0.986425,"{'C': 9.192798889964653, 'gamma': 0.0008664791..."
1,densenet121,LogReg,0.968445,0.972909,0.975436,{'C': 0.05423185466354641}
2,densenet121,SGD,0.956834,0.961193,0.965094,"{'alpha': 1.1676350874141216e-05, 'loss': 'mod..."


In [ ]:
# ============================================================
# [10c] Optuna diagnostics, default vs. tuned, and the final (embedding, model, parameters) choice
# READS : tuned_df, studies, baseline_all (in memory)
# WRITES: config/best_ml_params.json
# ============================================================
for (emb, name), study in studies.items():
    optuna.visualization.plot_optimization_history(study).update_layout(title=f"{emb} / {name}: optimisation history").show()
    try:
        optuna.visualization.plot_param_importances(study).update_layout(title=f"{emb} / {name}: parameter importance").show()
    except Exception as e:                                  # needs at least two completed trials
        print(emb, name, "- parameter importance unavailable:", e)

# Default (baseline) vs. tuned validation macro-F1
base = (baseline_all[baseline_all["strategy"] == TUNING_STRATEGY][["embedding", "model", "macro_f1"]]
        .rename(columns={"macro_f1": "default"}))
cmp_tuning = (tuned_df[["embedding", "model", "val_macro_f1"]].rename(columns={"val_macro_f1": "tuned"})
              .merge(base, on=["embedding", "model"], how="left"))
cmp_tuning["label"] = cmp_tuning["embedding"] + " / " + cmp_tuning["model"]
fig = px.bar(cmp_tuning.melt(id_vars=["label", "embedding", "model"], value_vars=["default", "tuned"],
                             var_name="version", value_name="val_macro_f1"),
             x="label", y="val_macro_f1", color="version", barmode="group",
             title="Validation macro-F1: default vs. Optuna-tuned parameters")
fig.update_yaxes(range=[max(0, cmp_tuning[["default", "tuned"]].min().min() - 0.02), 1.0])
fig.show()

# The single best combination, by validation macro-F1
FINAL_EMB = tuned_df.loc[0, "embedding"]
BEST_NAME = tuned_df.loc[0, "model"]
BEST_PARAMS = tuned_df.loc[0, "best_params"]
print(f"SELECTED: embedding={FINAL_EMB} | model={BEST_NAME} | params={BEST_PARAMS}")
with open(CFG_DIR / "best_ml_params.json", "w") as f:
    json.dump({"embedding": FINAL_EMB, "model": BEST_NAME, "params": BEST_PARAMS,
               "val_macro_f1": float(tuned_df.loc[0, "val_macro_f1"])}, f, indent=2)

SELECTED: embedding=densenet121 | model=SVC_RBF | params={'C': 9.192798889964653, 'gamma': 0.0008664791133069102}


## 11. Final classical-ML model
The selected model is refitted on **train + validation** of its embedding (more data, same hyper-parameters, balanced sample weights recomputed on the combined labels).
It is not evaluated here; its only evaluation happens in section 13.

In [ ]:
# ============================================================
# [11] Refit the selected pipeline on train + validation
# READS : features/<FINAL_EMB>/*.npy, config/best_ml_params.json (in memory)
# WRITES: models/best_ml_model.joblib
# ============================================================
F_final = load_features(FINAL_EMB)
X_trval = np.vstack([F_final["train"], F_final["val"]])
y_trval = np.concatenate([y_train, y_val])
sw_trval = None if BEST_NAME in NO_SAMPLE_WEIGHT else compute_sample_weight("balanced", y_trval)

final_ml = build_pipeline(BEST_NAME, **BEST_PARAMS)
t0 = time.time()
fit_pipeline(final_ml, X_trval, y_trval, sw_trval)
ml_train_sec = time.time() - t0
joblib.dump({"pipeline": final_ml, "classes": CLASS_NAMES, "embedding": FINAL_EMB}, MODEL_DIR / "best_ml_model.joblib")
print(f"refit {BEST_NAME} on {FINAL_EMB} embeddings ({len(X_trval)} samples) in {ml_train_sec:.1f}s")
print("saved", MODEL_DIR / "best_ml_model.joblib")

refit SVC_RBF on densenet121 embeddings (17530 samples) in 12.4s
saved /content/drive/MyDrive/plantvillage_outputs/models/best_ml_model.joblib


In [ ]:
# ============================================================
# [13] Final ML evaluation on the TEST set (once)
# ============================================================
import joblib, numpy as np, pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import plotly.express as px
from IPython.display import display

OUT = Path("/content/drive/MyDrive/plantvillage_outputs")
RES_DIR = OUT / "results"
RES_DIR.mkdir(exist_ok=True, parents=True)

bundle = joblib.load(OUT / "models" / "best_ml_model.joblib")
pipe, CLASS_NAMES, EMB = bundle["pipeline"], bundle["classes"], bundle["embedding"]
print("embedding:", EMB, "| model:", type(pipe.named_steps["model"]).__name__)

test = pd.read_csv(OUT / "data" / "test.csv")
X_test = np.load(OUT / "features" / EMB / "X_test.npy")
assert len(X_test) == len(test), "X_test rows do not match test.csv rows"

cls_to_idx = {c: i for i, c in enumerate(CLASS_NAMES)}
y_test = test["label"].map(cls_to_idx).values

pred = np.asarray(pipe.predict(X_test)).ravel().astype(int)

acc = accuracy_score(y_test, pred)
mf1 = f1_score(y_test, pred, average="macro")
wf1 = f1_score(y_test, pred, average="weighted")
print(f"TEST accuracy={acc:.4f} | macro-F1={mf1:.4f} | weighted-F1={wf1:.4f}")

short = lambda s: s.replace("___", "_").replace("__", "_")
names = [short(c) for c in CLASS_NAMES]

report = pd.DataFrame(classification_report(
    y_test, pred, target_names=names, output_dict=True, zero_division=0)).T
display(report.round(4))

cm = confusion_matrix(y_test, pred, labels=range(len(CLASS_NAMES)))
cm_norm = cm / cm.sum(axis=1, keepdims=True)

px.imshow(cm, x=names, y=names, text_auto=True, color_continuous_scale="Blues",
          labels=dict(x="Predicted", y="True"),
          title="Confusion matrix (counts)").update_layout(height=750).show()
px.imshow(cm_norm, x=names, y=names, text_auto=".2f", color_continuous_scale="Blues",
          labels=dict(x="Predicted", y="True"),
          title="Confusion matrix (row-normalised)").update_layout(height=750).show()

pd.DataFrame([dict(embedding=EMB, model=type(pipe.named_steps["model"]).__name__,
                   test_accuracy=acc, test_macro_f1=mf1, test_weighted_f1=wf1)]
             ).to_csv(RES_DIR / "final_results_ml.csv", index=False)
report.to_csv(RES_DIR / "report_ml.csv")
print("saved results to", RES_DIR)

embedding: densenet121 | model: SVC
TEST accuracy=0.9948 | macro-F1=0.9939 | weighted-F1=0.9948


,precision,recall,f1-score,support
Pepper_bell_Bacterial_spot,1.0000,1.0000,1.0000,150.0000
Pepper_bell_healthy,1.0000,1.0000,1.0000,221.0000
Potato_Early_blight,1.0000,1.0000,1.0000,150.0000
Potato_Late_blight,0.9933,0.9933,0.9933,150.0000
Potato_healthy,1.0000,0.9565,0.9778,23.0000
Tomato_Bacterial_spot,1.0000,0.9969,0.9984,319.0000
Tomato_Early_blight,0.9933,0.9933,0.9933,150.0000
Tomato_Late_blight,0.9895,0.9930,0.9912,285.0000
Tomato_Leaf_Mold,0.9931,1.0000,0.9965,143.0000
Tomato_Septoria_leaf_spot,0.9962,0.9962,0.9962,265.0000


saved results to /content/drive/MyDrive/plantvillage_outputs/results


###Clean_Splited_Data(images) For DL

In [ ]:
# ============================================================
# [Hand-off] Export train / val / test images for the deep-learning team
# READS : data/train.csv, data/val.csv, data/test.csv, image files
# WRITES: deep_learning/{train,val,test}/<class>/*.jpg, csv files, class_names.json, README.txt
# ============================================================
import shutil, json
from concurrent.futures import ThreadPoolExecutor

DL_DIR = OUT / "deep_learning"
INCLUDE_AUGMENTED = False        # True = add the augmented copies to train/
SPLIT_NAMES = ["train", "val", "test"]

splits_dl = {s: pd.read_csv(DATA_DIR / f"{s}.csv") for s in SPLIT_NAMES}
if INCLUDE_AUGMENTED and (DATA_DIR / "train_extra.csv").exists():
    ex = pd.read_csv(DATA_DIR / "train_extra.csv")[["path", "label"]]
    splits_dl["train"] = pd.concat([splits_dl["train"][["path", "label"]], ex], ignore_index=True)

for s in SPLIT_NAMES:
    print(f"{s}: {len(splits_dl[s])} images expected")

missing = [p for d in splits_dl.values() for p in d["path"] if not Path(p).exists()]
assert not missing, f"{len(missing)} image files not found, e.g. {missing[0]}. Run the Colab setup cells again."

def copy_one(args):
    src, dst = args
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists() or dst.stat().st_size != Path(src).stat().st_size:   # skip finished files, redo broken ones
        shutil.copy2(src, dst)

# Copy ONE split at a time, so a disconnection never leaves val/test untouched without you noticing
rows = []
for split in SPLIT_NAMES:
    d = splits_dl[split]
    jobs = [(src, DL_DIR / split / lab / Path(src).name) for src, lab in zip(d["path"], d["label"])]
    assert len({str(j[1]) for j in jobs}) == len(jobs), f"duplicate file names inside {split}"
    with ThreadPoolExecutor(max_workers=16) as pool:
        list(tqdm(pool.map(copy_one, jobs), total=len(jobs), desc=f"copying {split}"))
    rows += [(split, lab, str(dst.relative_to(DL_DIR))) for (src, dst), lab in zip(jobs, d["label"])]
    print(f"done: {split}")

# Index files, class names, README
index = pd.DataFrame(rows, columns=["split", "label", "relative_path"])
index.to_csv(DL_DIR / "index.csv", index=False)
for s in SPLIT_NAMES:
    index[index["split"] == s].to_csv(DL_DIR / f"{s}.csv", index=False)

classes = sorted(index["label"].unique())
json.dump({"classes": classes, "class_to_idx": {c: i for i, c in enumerate(classes)}},
          open(DL_DIR / "class_names.json", "w"), indent=2)
(DL_DIR / "README.txt").write_text(
    f"PlantVillage - data for the deep learning team\n"
    f"Same stratified split as the ML part (70/15/15, SEED={SEED}). Do NOT re-split.\n"
    f"Folders: train/ val/ test/ -> one folder per class (torchvision ImageFolder).\n"
    f"Use val for tuning/early stopping and test ONLY ONCE at the end.\n"
    f"Classes: {len(classes)} (see class_names.json). Images are 256x256 RGB.\n"
    f"Normalisation: ImageNet mean/std {IMAGENET_MEAN} / {IMAGENET_STD}.\n"
    f"Augmented copies in train/: {INCLUDE_AUGMENTED}\n")

# Final verification: count what is really on disk for each split
print("\n--- verification ---")
for s in SPLIT_NAMES:
    n_files = sum(1 for f in (DL_DIR / s).rglob("*") if f.suffix.lower() in EXT)
    n_cls = len([x for x in (DL_DIR / s).iterdir() if x.is_dir()])
    status = "OK" if n_files == len(splits_dl[s]) else "INCOMPLETE - run this cell again"
    print(f"{s}: {n_files}/{len(splits_dl[s])} images | {n_cls} classes | {status}")
print("saved to:", DL_DIR)

train: 14436 images expected
val: 3094 images expected
test: 3094 images expected


copying train:   0%|          | 0/14436 [00:00<?, ?it/s]

done: train


copying val:   0%|          | 0/3094 [00:00<?, ?it/s]

done: val


copying test:   0%|          | 0/3094 [00:00<?, ?it/s]

done: test

--- verification ---
train: 14436/14436 images | 15 classes | OK
val: 3094/3094 images | 15 classes | OK
test: 3094/3094 images | 15 classes | OK
saved to: /content/drive/MyDrive/plantvillage_outputs/deep_learning


In [ ]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount("/content/drive")

Mounted at /content/drive
